In [1]:
%run ./config_api_acto

StatementMeta(, a2a842ee-b218-4202-9771-a8de9a02430d, 3, Finished, Available, Finished, True)

In [2]:
import requests
import pandas as pd

TOKEN = TOKEN_OSASCO

BASE = "https://actogestaoapi-gdhrfgdfc8bbe8hs.brazilsouth-01.azurewebsites.net"
url = f"{BASE}/api/RelatoriosEtapa/ObterTempoEtapaRelatorio"

payload = {
    "codCatalogos": [6903], # cod BD: 6683
    "dataInicio": "2024-01-01T00:00:00.000Z",
    "dataFim": "2026-03-25T03:00:00.000Z",
    "ativo": 1
}

headers = {
    "Accept": "application/json",
    "Content-Type": "application/json",
    "Authorization": f"Bearer {TOKEN}",
}

r = requests.post(url, json=payload, headers=headers, timeout=60)
r.raise_for_status()
data = r.json()

df = pd.DataFrame(data if isinstance(data, list) else data.get("data", []))

df = df.drop(columns=["notifications", "isValid", "codEtapa"]) # codEtapa está 0 para tudo

for col in df.filter(like="data").columns:
    df[col] = pd.to_datetime(df[col], format="ISO8601")

StatementMeta(, a2a842ee-b218-4202-9771-a8de9a02430d, 4, Finished, Available, Finished, False)

In [3]:
sdf = spark.createDataFrame(df)
(
    sdf
    .write.mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_carta_servicos_tempo_etapa")
)

StatementMeta(, a2a842ee-b218-4202-9771-a8de9a02430d, 5, Finished, Available, Finished, False)